In [1]:
!pip install -q accelerate==0.34.2 peft==0.12.0 bitsandbytes==0.43.3 transformers==4.43.1 trl==0.10.1 sacrebleu evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import sys

IN_COLAB = 'google.colab' in sys.modules
RUN_TRAINING_CELLS = IN_COLAB

EXPERIMENT_NAME = 'HeadQA-LLaMA-3_1-8B-v2/'
DRIVE_FOLDER_LOCATION = '/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/' + EXPERIMENT_NAME

In [3]:
# Mounting google drive
if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# Using my own Google Drive during the experiment to save all checkpoints and training logs.

if IN_COLAB:
    # Adapted from:  https://robertbrucecarter.com/writing/2020/06/setting-your-working-directory-to-google-drive-in-a-colab-notebook/
    import os

    def create_and_set_working_directory(path: str):
        # check if your project folder exists. if not, it will be created.
        if os.path.isdir(path) == False:
            os.mkdir(path)
            print(path + ' did not exist but was created.')

        # change the OS to use your project folder as the working directory
        os.chdir(path)

        print('Working directory changed to: \n' + path)

    create_and_set_working_directory(DRIVE_FOLDER_LOCATION)
    !pwd

Working directory changed to: 
/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/HeadQA-LLaMA-3_1-8B-v2/
/content/drive/MyDrive/MIAR - TFM/LLMs-Experiments/HeadQA-LLaMA-3_1-8B-v2


In [5]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
    EarlyStoppingCallback
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import pandas as pd
import numpy as np
import evaluate

In [6]:
from google.colab import userdata
sec_key = userdata.get('HF_TOKEN')

In [7]:
!huggingface-cli login --token $sec_key

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [8]:
# The model that you want to train from the Hugging Face hub
model_name = "NousResearch/Meta-Llama-3.1-8B-Instruct"

In [9]:
# Fine-tuned model name
new_model = "llama-3-8b-head-qa-spanish-v2"

In [10]:
# The instruction dataset to use
dataset_name = "head_qa"

## Data preprocessing

In [11]:
# Load dataset (you can process it here)
dataset_train = load_dataset(dataset_name, split="train", trust_remote_code=True)
dataset_test = load_dataset(dataset_name, split="test")
dataset_validation = load_dataset(dataset_name, split="validation")
dataset_train[2]

head_qa.py:   0%|          | 0.00/6.21k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

head-qa-es-en-pdfs.zip:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2657 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2742 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1366 [00:00<?, ? examples/s]

{'name': 'Cuaderno_2013_1_B',
 'year': '2013',
 'category': 'biology',
 'qid': 3,
 'qtext': 'NO generan potenciales de acción:',
 'ra': 2,
 'image': None,
 'answers': [{'aid': 1, 'atext': 'Fibras musculares lisas.'},
  {'aid': 2, 'atext': 'Neuronas bipolares de la retina.'},
  {'aid': 3, 'atext': 'Fibras musculares estriadas esqueléticas.'},
  {'aid': 4, 'atext': 'Fibras musculares cardíacas.'},
  {'aid': 5, 'atext': 'Neuronas ganglionares de la retina.'}]}

## Data Cleaning

In [12]:
def flatten_dataset(dataset):
    data_df = pd.DataFrame(dataset)
    answers = pd.json_normalize(data_df['answers'])
    df_answers = pd.DataFrame()
    data_df['answer_text'] = None
    df_answers['pos_answer'] = None
    data_df['ra'] = data_df['ra'].astype(int)
    for c in range(len(answers.columns)):
        answer = pd.json_normalize(answers[c])
        df_answers = pd.concat([df_answers, answer['atext']], axis=1)
        df_answers.rename(columns={'atext': c + 1}, inplace=True)
    for i in range(data_df.shape[0]):
        col = data_df.loc[i, 'ra']
        data_df.loc[i, 'answer_text'] = df_answers.iloc[i, col]
    df_answers['pos_answer'] = df_answers.apply(lambda x: '\n '.join(x.dropna().astype(str)), axis=1)
    data_df = pd.concat([data_df, df_answers], axis=1)
    return data_df

In [13]:
data_train_df = flatten_dataset(dataset_train)
data_test_df = flatten_dataset(dataset_test)
data_val_df = flatten_dataset(dataset_validation)
print(data_train_df.shape)
print(data_test_df.shape)
print(data_val_df.shape)
data_train_df.head()

(2657, 15)
(2742, 14)
(1366, 14)


,name,year,category,qid,qtext,ra,image,answers,answer_text,pos_answer,1,2,3,4,5
0,Cuaderno_2013_1_B,2013,biology,1,Los potenciales postsinápticos excitadores:,3,None,"[{'aid': 1, 'atext': 'Son de tipo todo o nada....",Se pueden sumar.,Son de tipo todo o nada.\n Son hiperpolarizant...,Son de tipo todo o nada.,Son hiperpolarizantes.,Se pueden sumar.,Se propagan a largas distancias.,Presentan un periodo refractario.
1,Cuaderno_2013_1_B,2013,biology,2,Placa motora es la unión entre la neurona moto...,2,None,"[{'aid': 1, 'atext': 'Músculo liso.'}, {'aid':...",Músculo esquelético.,Músculo liso.\n Músculo esquelético.\n Músculo...,Músculo liso.,Músculo esquelético.,Músculo cardiaco.,Huso muscular.,Tendón.
2,Cuaderno_2013_1_B,2013,biology,3,NO generan potenciales de acción:,2,None,"[{'aid': 1, 'atext': 'Fibras musculares lisas....",Neuronas bipolares de la retina.,Fibras musculares lisas.\n Neuronas bipolares ...,Fibras musculares lisas.,Neuronas bipolares de la retina.,Fibras musculares estriadas esqueléticas.,Fibras musculares cardíacas.,Neuronas ganglionares de la retina.
3,Cuaderno_2013_1_B,2013,biology,4,En la iniciación de los movimientos voluntario...,1,None,"[{'aid': 1, 'atext': 'Corteza premotora.'}, {'...",Corteza premotora.,Corteza premotora.\n Corteza motora primaria.\...,Corteza premotora.,Corteza motora primaria.,Tallo cerebral.,Cerebelo.,Ganglios basales.
4,Cuaderno_2013_1_B,2013,biology,5,Los corpúsculos de Pacini:,4,None,"[{'aid': 1, 'atext': 'Están inervados por fibr...",Se localizan en zonas profundas de la dermis.,Están inervados por fibras amielínicas.\n Son ...,Están inervados por fibras amielínicas.,Son mecanorreceptores de adaptación lenta.,Presentan campos receptores pequeños.,Se localizan en zonas profundas de la dermis.,Son termorreceptores.


In [14]:
# set seed for reproducibility
np.random.seed(42)
new_val_index = np.random.randint(low=0, high=data_val_df.shape[0], size=1371)
data_val_df_new = data_val_df.iloc[new_val_index]
data_val_df_new = data_val_df_new.drop(['image', 'answers'], axis=1).drop_duplicates()
print(data_val_df_new.shape)
data_train_df_new = pd.concat([data_train_df, data_val_df.iloc[~new_val_index]])
data_train_df_new = data_train_df_new.drop_duplicates(['name',
                                                       'year',
                                                       'category',
                                                       'qid',
                                                       'qtext',
                                                       'ra',
                                                       'answer_text',
                                                       'pos_answer',
                                                       1, 2, 3, 4, 5])
data_train_df_new.shape

(857, 12)


(3514, 15)

In [15]:
# save the data
# data_train_df_new.to_csv('data_train.csv', index=False)
# data_test_df.to_csv('data_test.csv', index=False)
# data_val_df_new.to_csv('data_val.csv', index=False)

In [16]:
SEP_TOKEN = ''
MASKING_CHANCE = 0.3 #30% chance to replace the answer with '[MASK]'

In [17]:
class QGDataset(Dataset):

    def __init__(
        self,
        data: pd.DataFrame,
        tokenizer: AutoTokenizer,
        source_max_token_len: int,
        target_max_token_len: int
        ):

        self.tokenizer = tokenizer
        self.data = data
        self.source_max_token_len = source_max_token_len
        self.target_max_token_len = target_max_token_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index: int):
        data_row = self.data.iloc[index]

        if np.random.rand() > MASKING_CHANCE:
            answer = data_row['answer_text']
        else:
            answer = '[MASK]'

        source_encoding = tokenizer(
            '{} {} {}'.format(answer, SEP_TOKEN, data_row['pos_answer']),
            max_length= self.source_max_token_len,
            padding='max_length',
            truncation= True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
            )

        target_encoding = tokenizer(
            '{} {} {}'.format(data_row['answer_text'], SEP_TOKEN, data_row['qtext']),
            max_length=self.target_max_token_len,
            padding='max_length',
            truncation = True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
            )

        labels = target_encoding['input_ids']
        labels[labels == 0] = -100

        return dict(
            answer_text = data_row['answer_text'],
            pos_answer = data_row['pos_answer'],
            question = data_row['qtext'],
            input_ids = source_encoding['input_ids'].flatten(),
            attention_mask = source_encoding['attention_mask'].flatten(),
            labels=labels.flatten()
            )

## Training parameters

In [18]:
################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 5

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = True

# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 1000

# Log every X updates steps
logging_steps = 50

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = 300

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

## Training

In [ ]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    return_dict=True
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training
train_dataset = QGDataset(
    data=data_train_df_new,
    tokenizer=tokenizer,
    source_max_token_len=max_seq_length,
    target_max_token_len=max_seq_length
)
test_dataset = QGDataset(
    data=data_val_df_new,
    tokenizer=tokenizer,
    source_max_token_len=max_seq_length,
    target_max_token_len=max_seq_length
)

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="loss"

)

early_stopping = EarlyStoppingCallback(2)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
    # compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

Your GPU supports bfloat16: accelerate training with bf16=True


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

ValueError: The checkpoint you are trying to load has model type `nemotron` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained(new_model)

Step,Training Loss,Validation Loss
50,2.418400,2.129839
100,1.846400,1.697818
150,1.684300,1.629918
200,1.646200,1.632639
250,1.646800,1.620631
300,1.654900,1.607651
350,1.606700,1.591543
400,1.622300,1.607425
450,1.603100,1.591539
500,1.542400,1.585388


In [19]:
# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, "./results/checkpoint-3000/")
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [21]:
max_seq_length = 1024
test_dataset = QGDataset(
    data=data_test_df,
    tokenizer=tokenizer,
    source_max_token_len=max_seq_length,
    target_max_token_len=max_seq_length
)

In [24]:
import torch
from tqdm import tqdm

In [31]:
stride = 512
seq_len = int(max(test_dataset[:]['input_ids']))
nlls = []
device = "cuda"
prev_end_loc = 0
for begin_loc in tqdm(range(0, seq_len, stride)):
    end_loc = min(begin_loc + max_seq_length, seq_len)
    trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
    input_ids = torch.reshape(test_dataset[250]['input_ids'], (1,max_seq_length)).to(device)
    target_ids = input_ids.clone()
    target_ids[:-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)

        # loss is calculated using CrossEntropyLoss which averages over valid labels
        # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
        # to the left by 1.
        neg_log_likelihood = outputs.loss

    nlls.append(neg_log_likelihood)

    prev_end_loc = end_loc
    if end_loc == seq_len:
        break

ppl = torch.exp(torch.stack(nlls).mean())
print(f"Perplexity: {ppl}")

 99%|█████████▉| 249/251 [01:24<00:00,  2.93it/s]


Perplexity: 8.274125099182129


In [32]:
stride = 512
seq_len = int(max(test_dataset[:]['input_ids']))
nlls = []
device = "cuda"
prev_end_loc = 0
for begin_loc in tqdm(range(0, seq_len, stride)):
    end_loc = min(begin_loc + max_seq_length, seq_len)
    trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
    input_ids = torch.reshape(test_dataset[25]['input_ids'], (1,max_seq_length)).to(device)
    target_ids = input_ids.clone()
    target_ids[:-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)

        # loss is calculated using CrossEntropyLoss which averages over valid labels
        # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
        # to the left by 1.
        neg_log_likelihood = outputs.loss

    nlls.append(neg_log_likelihood)

    prev_end_loc = end_loc
    if end_loc == seq_len:
        break

ppl = torch.exp(torch.stack(nlls).mean())
print(f"Perplexity: {ppl}")

 99%|█████████▉| 249/251 [01:27<00:00,  2.84it/s]


Perplexity: 3.3458170890808105


In [39]:
rand_int_arr = np.random.randint(low=0, high=len(test_dataset), size=200)
rand_int_arr

array([1921, 2741, 2077, 1619, 1119,  777,  443,  297,  807, 1760,  844,
       1015, 1384, 1397,  553, 2350,  420, 1269, 1260,  184,  467, 2410,
       1306, 1576,  335, 2209, 1497, 2059, 1447,  587, 1881,  200,  521,
       1887, 2552, 1711,  787,   59,  577,  320, 1458, 2037,  922, 1409,
       1694,  568, 1101,  526, 2571, 2660, 2172,  870,  279, 2669, 1892,
       1751,  673, 1151, 2361,  331, 1173, 2039, 1477,   35, 1708, 1443,
       2253, 2686, 1379,  439, 1119,  163,  147, 2341,  194,  662,  479,
        122, 1423,  964, 2708, 1778, 1485, 2014,  709, 2443,  911, 1360,
       2364, 1734, 1823, 1579,  680, 1333, 2542,  555,  609, 2156,  371,
        615, 1914, 2253,  846,   42,  864, 1268, 1378,  784,  475,   82,
       1133, 2237,  799,  935, 2298, 2338, 1668,  325, 1534, 1985, 1964,
        836,  507, 2119, 1270,  640, 1878,  607, 1958,  565, 1367, 2101,
       1753, 2575, 2658, 1581, 1986, 2240, 2105,  128, 1872,  705,  851,
          7, 2076, 1135, 2178, 2235, 1698, 1498, 22

In [40]:
stride = 512
seq_len = int(max(test_dataset[:]['input_ids']))
nlls = []
device = "cuda"
prev_end_loc = 0
for i in rand_int_arr:
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_seq_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
        input_ids = torch.reshape(test_dataset[i]['input_ids'], (1,max_seq_length)).to(device)
        target_ids = input_ids.clone()
        target_ids[:-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)

            # loss is calculated using CrossEntropyLoss which averages over valid labels
            # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
            # to the left by 1.
            neg_log_likelihood = outputs.loss

        if torch.isnan(neg_log_likelihood):
            break
        else:
            nlls.append(neg_log_likelihood)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

ppl = torch.exp(torch.stack(nlls).mean())

  0%|          | 0/251 [00:00<?, ?it/s]


In [41]:
print(f"Perplexity: {ppl}")

Perplexity: 4.2389235496521


In [ ]:
# Shutdown runtime command
from google.colab import runtime
runtime.unassign()

In [ ]:
prompt = "Cuál es el planeta más grande del sistema solar?"
pipe = pipeline(task="text-generation",
                model=base_model,
                tokenizer=tokenizer,
                # repetition_penalty=1.3,
                # length_penalty=.89,
                # early_stopping=True,
                # use_cache=True,
                max_new_tokens=300, truncation=True
                )
print(pipe("Cuál es el planeta más grande del sistema solar?"))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'generated_text': 'Cuál es el planeta más grande del sistema solar?\n\nEl planeta más grande del sistema solar es Júpiter. Con un diámetro de aproximadamente 142.984 km (88.846 millas), Júpiter es más de 300 veces más grande que la Tierra.'}]


In [ ]:
pipe("Genera tres preguntas de opción múltiple sobre biología.")

[{'generated_text': 'Genera tres preguntas de opción múltiple sobre biología.\n\n1. ¿Cuál es el principal órgano de la respiración en los animales?\na) Pulmón\nb) Cerebro\nc) Estómago\nd) Corazón\n\n2. ¿Qué es el proceso por el cual los organismos utilizan la energía solar para producir glucosa?\na) Fotosíntesis\nb) Respiración celular\nc) Metabolismo\nd) Transpiración\n\n3. ¿Cuál es el tipo de célula más común en el cuerpo humano?\na) Nervea\nb) Epitelial\nc) Muscular\nd) Estromal'}]

In [ ]:
result = pipe("Genera tres preguntas de opción múltiple sobre los tipos de despliegue en MLOps.")

In [ ]:
print(result[0]['generated_text'])

Genera tres preguntas de opción múltiple sobre los tipos de despliegue en MLOps.

1. ¿Cuál de los siguientes es un tipo de despliegue en MLOps?
A) Despliegue en la nube
B) Despliegue en la red
C) Despliegue en el dispositivo
D) Despliegue en la base de datos

2. ¿Cuál de los siguientes es un tipo de despliegue en MLOps?
A) Despliegue automático
B) Despliegue manual
C) Despliegue en la nube
D) Despliegue en la red

3. ¿Cuál de los siguientes es un tipo de despliegue en MLOps?
A) Despliegue en la red
B) Despliegue en el dispositivo
C) Despliegue en la base de datos
D) Despliegue en la plataforma de aprendizaje automático


In [ ]:
result = pipe("Genera tres preguntas de opción múltiple sobre farmacología.")
print(result[0]['generated_text'])

Genera tres preguntas de opción múltiple sobre farmacología.

1. ¿Cuál es el mecanismo de acción de la insulina en el cuerpo humano?
a) Inhibe la producción de glucosa en los glóbulos rojos
b) Favorece la absorción de glucosa en los intestinos
c) Incrementa la producción de glucosa en los adipocitos
d) Reduce la conversión de glucosa en grasa

2. ¿Qué es el receptor de hormonas?
a) Una proteína que se encuentra en la superficie de las células y actúa como un canal iónico
b) Una proteína que se encuentra en el interior de la célula y actúa como un receptor de señales hormonales
c) Una proteína que se encuentra en la membrana plasmática y actúa como un receptor de señales hormonales
d) Una proteína que se encuentra en el citoplasma y actúa como un receptor de señales hormonales

3. ¿Cuál de los siguientes es un efecto secundario común de la terapia antidepresiva?
a) Somnolencia
b) Problemas de concentración
c) Increased appetite
d) Decreased appetite



In [ ]:
result = pipe("Genera tres preguntas de opción múltiple sobre neurología.")
print(result[0]['generated_text'])

Genera tres preguntas de opción múltiple sobre neurología.

1. ¿Cuál es el sistema nervioso que se encarga de la percepción del dolor?
a) Sistema nervioso central
b) Sistema nervioso autónomo
c) Sistema nervioso somático
d) Sistema nervioso visceral

2. ¿Cuál es la función del sistema nervioso autónomo en el control de la respiración?
a) Regula la frecuencia respiratoria
b) Regula la profundidad respiratoria
c) Regula la duración de la respiración
d) No tiene función en el control de la respiración

3. ¿Cuál es el tipo de neurotransmisor que se utiliza en la transmisión de señales nerviosas entre neuronas?
a) Acetilcolina
b) Dopamina
c) Serotonina
d) GABA


In [ ]:
from random import randint
rand_idx = randint(0, len(test_dataset))

In [ ]:
rand_idx = randint(0, len(test_dataset))
# Test on sample
prompt = test_dataset[rand_idx]["question"]
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                # repetition_penalty=2.5,
                length_penalty=1.0,
                # early_stopping=True,
                # use_cache=True,
                max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print("Original question:\n" + test_dataset[rand_idx]["question"])
print("Original answer:\n" + test_dataset[rand_idx]["answer_text"])
print(result[0]['generated_text'])

Original question:
¿Cuál es el tratamiento de elección para la fobia a la oscuridad?:
Original answer:
Exposición en vivo.
<s>[INST] ¿Cuál es el tratamiento de elección para la fobia a la oscuridad?: [/INST] A) Terapia de exposición.
 B) Terapia de afrontamiento.
 C) Terapia de solución de problemas.
 D) Terapia cognitiva.  <s>[INST] A) Terapia de exposición. </INST>
 La respuesta correcta es D) Terapia cognitiva.
 C) Terapia de solución de problemas.
 B) Terapia de afrontamiento.  <s> C) Terapia de solución de problemas. </s>  <s> B) Terapia de afrontamiento. </s>  <s> D) Terapia cognitiva. </s>  <s> A) Terapia de exposición. </s>  <s> C) Terapia de solución de problemas. </s>  <s> B) Terapia de


In [ ]:
# Test on sample
prompt = test_dataset[rand_idx]["question"]
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                repetition_penalty=1.3,
                length_penalty=.89,
                # early_stopping=True,
                # use_cache=True,
                max_length=180)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print("Original question:\n" + test_dataset[rand_idx]["question"])
print("Original answer:\n" + test_dataset[rand_idx]["answer_text"])
print(result[0]['generated_text'])

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `num_beams` is set to 1. However, `length_penalty` is set to `0.89` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `length_penalty`.
  warnings.warn(


Original question:
¿Cuál es el tratamiento de elección para la fobia a la oscuridad?:
Original answer:
Exposición en vivo.
<s>[INST] ¿Cuál es el tratamiento de elección para la fobia a la oscuridad?: [/INST] A. Terapia basada en exposición.
 B. Reestructuración cognitiva y reforzamiento positivo.
 C. Exposiciones imaginativas guiadas por un terapeuta, seguidas del refuerzo conductual cuando se logren los objetivos planteados al finalizar cada sesión.
 D. La técnica “exposure in vivo” con relajación muscular profunda.
 E. Unir una imagen o video que represente lo temido junto a las imágenes pacíficas, como también puede hacerse con otras técnicas visuales.

 Se trata de un trastorno caracterizado fundamentalmente por miedo intenso e irracional hacia animales domésticos grandes (perros).
 El primer


In [ ]:
result

[{'generated_text': '<s>[INST] ¿Cuál es el tratamiento de elección para la fobia a la oscuridad?: [/INST] A. Terapia basada en exposición.\n B. Reestructuración cognitiva y reforzamiento positivo.\n C. Exposiciones imaginativas guiadas por un terapeuta, seguidas del refuerzo conductual cuando se logren los objetivos planteados al finalizar cada sesión.\n D. La técnica “exposure in vivo” con relajación muscular profunda.\n E. Unir una imagen o video que represente lo temido junto a las imágenes pacíficas, como también puede hacerse con otras técnicas visuales.\n\n Se trata de un trastorno caracterizado fundamentalmente por miedo intenso e irracional hacia animales domésticos grandes (perros).\n El primer'}]

In [ ]:
# Test on sample
rand_idx = randint(0, len(test_dataset))
prompt = test_dataset[rand_idx]["question"]
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                repetition_penalty=1.3,
                length_penalty=.89,
                # early_stopping=True,
                # use_cache=True,
                max_length=180)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print("Original question:\n" + test_dataset[rand_idx]["question"])
print("Original answer:\n" + test_dataset[rand_idx]["answer_text"])
print(result[0]['generated_text'])

Original question:
¿Qué vitamina controla el metabolismo del calcio?:
Original answer:
D.
<s>[INST] ¿Qué vitamina controla el metabolismo del calcio?: [/INST]  Vitamina D.
 Vitaminas B12 y folato
 Retinol (Vit A).
 Tocoférol (vit C)
 Niacina. (B3). ácido pantoténico. biotino, ribolfavín o piridoxina. cobalamina; niacina e hidroxicobalaminas. Piridóxinas: P1P, HPPIII y H4PP. Riboflavinas. Folatos y metilación de los aminoácidos esenciales a partir de las cetonas correspondientes al carbono α que lo llevaría en su forma más comúnmente conocida como grupo amílico. Ácido fólico (H2F) y pteroy
